# 响应与错误处理

学习目标：为记录查询定义公开响应，区分正常结果、请求错误与程序错误，并让错误内容和 OpenAPI 声明一致。

前置知识：HTTP 请求与响应、Python 函数、类型标注、字典、异常处理、Pydantic 模型。

运行环境：Python 3.12、Pydantic v2；完整依赖版本见环境入口。

环境准备：[FastAPI 环境与运行入口](README.md)。

工作目录： content/Web与应用开发/FastAPI；从空内核自上而下运行。本章使用 TestClient 作应用内调用，并读取 OpenAPI 文档，不启动监听端口。

## 1 用返回类型说明公开结果

查询一条记录时，调用方需要知道结果包含哪些字段。返回类型标注为 Pydantic 模型后，FastAPI 会校验返回数据、序列化为 JSON，并在 OpenAPI 中描述响应结构。OpenAPI 是供文档工具和客户端工具读取的接口说明。

先只返回一条固定记录。TestClient 直接调用应用，返回对象的 status_code 和 json() 分别用于观察 HTTP 状态码与 JSON 内容。

In [1]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel


class RecordOutput(BaseModel):
    id: int
    title: str


app = FastAPI()


@app.get("/preview")
def preview_record() -> RecordOutput:
    return RecordOutput(id=1, title="阅读 HTTP 文档")


with TestClient(app) as client:
    response = client.get("/preview")
    print(response.status_code, response.json())  # 预期：200 {'id': 1, 'title': '阅读 HTTP 文档'}。
    # 正常响应为 200，JSON 对象包含模型声明的两个字段。
    assert response.json() == {"id": 1, "title": "阅读 HTTP 文档"}

200 {'id': 1, 'title': '阅读 HTTP 文档'}


C:\Users\ZHUANG\miniconda3\envs\hands-on-computing\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## 2 查询记录：过滤内部字段，找不到时返回 404

内部字典保留 id、title 与 private\_note，公开模型只声明前两项。response\_model 指定返回内容的公开边界，即使函数返回完整字典，响应仍过滤额外字段；同时存在返回类型时，以 response\_model 为准。

![成功结果与业务错误走不同出口](image/illustration/04-01-response-boundary.svg)

图示：本例的公开响应边界。上方为正常返回后的响应模型处理，下方为业务异常响应。

路径参数 record\_id 是整数编号。编号不存在时，raise HTTPException 终止查询并返回 404，默认错误内容放在 detail 中，不能把异常对象当正常结果 return。

下面分别请求 1、999：检查成功响应不含 private\_note，同时原字典仍保留它；再核对缺失记录的状态码与错误内容。

In [2]:
from fastapi import HTTPException


records = {
    1: {"id": 1, "title": "阅读 HTTP 文档", "private_note": "虚构的内部备注"}
}


@app.get("/records/{record_id}", response_model=RecordOutput)
def read_record(record_id: int) -> dict[str, object]:
    if record_id not in records:
        raise HTTPException(status_code=404, detail="记录不存在")
    return records[record_id]


with TestClient(app) as client:
    found = client.get("/records/1")
    missing = client.get("/records/999")
    print(found.status_code, found.json())  # 预期：200 {'id': 1, 'title': '阅读 HTTP 文档'}。
    print(missing.status_code, missing.json())  # 预期：404 {'detail': '记录不存在'}。
    # 返回的 JSON 经过过滤；内部字典本身仍保留备注。
    assert "private_note" in records[1]
    assert "private_note" not in found.json()
    assert missing.status_code == 404
    assert missing.json() == {"detail": "记录不存在"}

200 {'id': 1, 'title': '阅读 HTTP 文档'}
404 {'detail': '记录不存在'}


## 3 请求校验失败发生在查询之前

不存在的整数编号可以进入查询函数，再由代码决定返回 404。无法解析成整数的路径参数则触发 RequestValidationError，FastAPI 默认返回 422，查询函数不会收到这个无效参数。

默认错误列表中的 loc 指出参数位置，type 是错误类别。观察这两个字段，比依赖一整句错误提示文本更容易看清失败原因。

In [3]:
with TestClient(app) as client:
    invalid = client.get("/records/not-an-integer")
    issue = invalid.json()["detail"][0]
    print(invalid.status_code, issue["loc"], issue["type"])  # 预期：422 ['path', 'record_id'] int_parsing。
    # path 表示路径参数；此次失败还没有进入 records 字典查询。
    assert invalid.status_code == 422
    assert issue["loc"] == ["path", "record_id"]
    assert issue["type"] == "int_parsing"

422 ['path', 'record_id'] int_parsing


## 4 创建记录：输入模型、状态码和响应头

输入模型描述调用方可以提交什么，输出模型描述调用方可以看到什么。这里允许提交内部备注，但创建结果仍使用 RecordOutput，避免将输入原样公开。

status_code=201 声明创建成功的状态码。接收一个 Response 参数可以设置响应头，函数仍返回普通数据，响应模型的校验与过滤照常执行。下面用 Location 告诉调用方新记录的地址；数据只保存在本次内核的字典中。

In [4]:
from fastapi import Response


class RecordInput(BaseModel):
    title: str
    private_note: str


@app.post("/records", response_model=RecordOutput, status_code=201)
def create_record(record: RecordInput, response: Response) -> dict[str, object]:
    record_id = max(records) + 1
    # model_dump() 将模型转成字典；下一节比较它的不同表示方式。
    records[record_id] = {"id": record_id, **record.model_dump()}
    response.headers["Location"] = f"/records/{record_id}"
    return records[record_id]


payload = {"title": "练习响应", "private_note": "仅用于本次演示"}
with TestClient(app) as client:
    created = client.post("/records", json=payload)
    loaded = client.get(created.headers["Location"])
    # 预期：201 /records/2 {'id': 2, 'title': '练习响应'}。
    print(created.status_code, created.headers["Location"], created.json())
    # 响应头指向可查询的记录；创建与查询都没有公开内部备注。
    assert created.status_code == 201
    assert created.json() == loaded.json() == {"id": 2, "title": "练习响应"}

201 /records/2 {'id': 2, 'title': '练习响应'}


## 5 区分 Python 数据、JSON 兼容数据和 JSON 文本

普通模型的 model_dump() 返回 Python 字典，其中可能仍有 datetime 等对象。model_dump(mode="json") 返回可用 JSON 表示的数据；model_dump_json() 直接生成 JSON 文本字符串。

这三者描述的是同一份数据的不同表示。普通路由可以直接返回模型或字典，由 FastAPI 完成响应序列化；主动构造响应对象时，需要选择正确的表示。

In [5]:
from datetime import datetime, timezone
import json


class DatedRecord(BaseModel):
    title: str
    created_at: datetime


dated = DatedRecord(
    title="保存阅读时间", created_at=datetime(2026, 9, 15, 8, 30, tzinfo=timezone.utc)
)
python_data = dated.model_dump()
json_data = dated.model_dump(mode="json")
json_text = dated.model_dump_json()
print(type(python_data["created_at"]).__name__)  # 预期：datetime。
print(json_data)  # 预期：{'title': '保存阅读时间', 'created_at': '2026-09-15T08:30:00Z'}。
print(json_text)  # 预期：{"title":"保存阅读时间","created_at":"2026-09-15T08:30:00Z"}。
# JSON 兼容字典中的时间已变成字符串；JSON 文本解析后得到同一字典。
assert isinstance(python_data["created_at"], datetime)
assert isinstance(json_data["created_at"], str)
assert json.loads(json_text) == json_data

datetime
{'title': '保存阅读时间', 'created_at': '2026-09-15T08:30:00Z'}
{"title":"保存阅读时间","created_at":"2026-09-15T08:30:00Z"}


## 6 用异常处理器统一错误内容

异常处理器（exception handler）接收请求和指定类型的异常，并返回一个响应。下面用 contract_app 单独演示自定义格式，保留 app 的默认错误格式作观察对照。

先定义含 code 和 message 的错误模型，并处理 HTTPException。处理器注册到 StarletteHTTPException，因为 FastAPI 的 HTTPException 继承它；这样也能处理框架产生的 HTTP 错误。本例的 detail 使用文本，保留异常携带的状态码和响应头。

In [6]:
from fastapi import Request
from fastapi.responses import JSONResponse
from starlette.exceptions import HTTPException as StarletteHTTPException


class ErrorOutput(BaseModel):
    code: str
    message: str


contract_app = FastAPI(title="记录响应示例")


@contract_app.exception_handler(StarletteHTTPException)
async def handle_http_error(request: Request, exc: StarletteHTTPException):
    error = ErrorOutput(code=f"HTTP_{exc.status_code}", message=str(exc.detail))
    return JSONResponse(
        status_code=exc.status_code,
        content=error.model_dump(mode="json"),
        headers=exc.headers,
    )

请求校验异常需要自己的处理器。这里保留 422 状态码，只返回固定错误码和简短提示；不要直接把异常的完整字符串发给调用方，它可能包含文件位置等内部信息。

JSONResponse 接收 JSON 兼容数据，负责构造响应内容。先注册两个处理器，再发出请求；后面为 contract_app 添加查询路由。

In [7]:
from fastapi.exceptions import RequestValidationError


@contract_app.exception_handler(RequestValidationError)
async def handle_request_error(request: Request, exc: RequestValidationError):
    error = ErrorOutput(code="INVALID_REQUEST", message="请求参数不符合要求")
    return JSONResponse(status_code=422, content=error.model_dump(mode="json"))


with TestClient(contract_app) as client:
    unknown = client.get("/unknown")
    print(unknown.status_code, unknown.json())  # 预期：404 {'code': 'HTTP_404', 'message': 'Not Found'}。
    # 尚未定义业务路由；未知路径也会经过 HTTP 异常处理器。
    assert unknown.status_code == 404
    assert unknown.json() == {"code": "HTTP_404", "message": "Not Found"}

404 {'code': 'HTTP_404', 'message': 'Not Found'}


## 7 为实际错误声明 OpenAPI 模型与示例

responses 按状态码补充响应模型、描述和示例。它只描述接口契约，不会自动替处理器生成错误内容，也不会把处理器的 JSONResponse 再套进所声明的模型。

下面给 contract_app 添加查询入口，复用 read_record 的查询逻辑。200 仍由 RecordOutput 描述；404 和自定义的 422 都声明 ErrorOutput。content 中的 application/json 表示 JSON 媒体类型，example 是文档中的示例数据。

In [8]:
missing_example = {"code": "HTTP_404", "message": "记录不存在"}
invalid_example = {"code": "INVALID_REQUEST", "message": "请求参数不符合要求"}
# 文档按状态码分别描述正常响应、缺失记录和请求校验失败。
response_docs = {
    200: {
        "description": "返回公开记录",
        "content": {"application/json": {"example": {"id": 1, "title": "阅读 HTTP 文档"}}},
    },
    404: {
        "model": ErrorOutput,
        "description": "记录不存在",
        "content": {"application/json": {"example": missing_example}},
    },
    422: {
        "model": ErrorOutput,
        "description": "请求参数不符合要求",
        "content": {"application/json": {"example": invalid_example}},
    },
}


# responses 描述接口文档；实际查询和报错仍由下面调用的 read_record 执行。
@contract_app.get(
    "/records/{record_id}", response_model=RecordOutput, responses=response_docs
)
def documented_record(record_id: int) -> dict[str, object]:
    return read_record(record_id)

用实际请求核对三个状态码，再读取 /openapi.json 检查声明。模型以 JSON Schema 描述；其中的 $ref 表示文档内部引用，下面确认它指向 ErrorOutput。

文档示例、实际返回和模型需要一起维护，修改异常处理器并不会自动改写 responses 中的错误声明。

In [9]:
# 1. 分别取得三种实际响应和应用生成的 OpenAPI 文档。
with TestClient(contract_app) as client:
    success = client.get("/records/1")
    missing = client.get("/records/999")
    invalid = client.get("/records/not-an-integer")
    schema = client.get("/openapi.json").json()
    for response in (success, missing, invalid):
        print(response.status_code, response.json())  # 预期：三次依次为 200 的记录、404 的 HTTP_404、422 的 INVALID_REQUEST 错误对象。
    # 文档中的示例与这三个具体请求的结果保持一致。
    statuses = [success.status_code, missing.status_code, invalid.status_code]
    assert statuses == [200, 404, 422]
    assert missing.json() == missing_example
    assert invalid.json() == invalid_example

# 2. 将文档中的示例、模型引用与对应响应对照。
declared = schema["paths"]["/records/{record_id}"]["get"]["responses"]
print("文档状态码：", sorted(declared))  # 预期：['200', '404', '422']。
for status_code, response in (("200", success), ("404", missing), ("422", invalid)):
    media = declared[status_code]["content"]["application/json"]
    assert media["example"] == response.json()
for status_code in ("404", "422"):
    media = declared[status_code]["content"]["application/json"]
    assert media["schema"]["$ref"] == "#/components/schemas/ErrorOutput"
    ErrorOutput.model_validate(media["example"])
# 公开模型没有内部备注；此处核对的是文档结构与真实应用内响应。
public_fields = schema["components"]["schemas"]["RecordOutput"]["properties"]
assert set(public_fields) == {"id", "title"}

200 {'id': 1, 'title': '阅读 HTTP 文档'}
404 {'code': 'HTTP_404', 'message': '记录不存在'}
422 {'code': 'INVALID_REQUEST', 'message': '请求参数不符合要求'}
文档状态码： ['200', '404', '422']


## 8 响应模型不合格属于程序错误

查询失败的 404 是接口主动表达的业务结果，422 是请求不符合参数要求。函数返回的数据缺少响应模型必填字段，则说明应用没有履行自身的返回约定：FastAPI 抛出 ResponseValidationError，客户端得到 500。

用单独的 boundary_app 保留一个故意写错的路由。TestClient 默认把应用异常抛给测试代码，便于定位；设置 raise_server_exceptions=False 后可以观察客户端收到的 500 响应。不要把这样的程序错误统统转换成请求校验错误。

In [10]:

boundary_app = FastAPI()


@boundary_app.get("/broken", response_model=RecordOutput)
def broken_record() -> dict[str, int]:
    return {"id": 1}  # 故意缺少 title，观察响应校验失败。

先用默认 TestClient 直接观察原始 ResponseValidationError：位置为 response/title。该反例运行报错后继续下一单元，比较真正的 HTTP 错误响应。

In [11]:
# 预期 ResponseValidationError：返回字典缺少响应模型必填的 title。
with TestClient(boundary_app) as client:
    client.get("/broken")

ResponseValidationError: 1 validation error:
  {'type': 'missing', 'loc': ('response', 'title'), 'msg': 'Field required', 'input': {'id': 1}}

  File "C:\Users\ZHUANG\AppData\Local\Temp\ipykernel_38288\2163000325.py", line 4, in broken_record
    GET /broken

下面仅为观察客户端的 500 响应设置 raise_server_exceptions=False；不修改应用，也不将程序错误转换为成功结果。

In [12]:
with TestClient(boundary_app, raise_server_exceptions=False) as client:
    failure = client.get("/broken")
    print(failure.status_code, failure.text)  # 预期：500 Internal Server Error。
    assert failure.status_code == 500

500 Internal Server Error


## 9 直接返回 Response 时自行负责内容

Response 及其子类已经是响应对象。直接返回 JSONResponse 时，FastAPI 会直接发送它，跳过响应模型的数据校验、转换与字段过滤。这也解释了为什么异常处理器要自己构造 ErrorOutput。

下面是刻意保留的边界反例：即使声明 RecordOutput，直接响应中的错误 id 和额外字段仍能发出去。业务数据通常直接返回模型或字典；只有需要控制完整响应时才主动构造 Response，并自行保证内容与声明一致。

In [13]:
@boundary_app.get("/direct", response_model=RecordOutput)
def direct_record() -> JSONResponse:
    return JSONResponse(content={"id": "不是整数", "internal_note": "虚构演示字段"})


with TestClient(boundary_app) as client:
    direct = client.get("/direct")
    print(direct.status_code, direct.json())  # 预期：200 {'id': '不是整数', 'internal_note': '虚构演示字段'}。
    # 这是绕过响应模型的反例，不能依赖 response_model 自动保护此响应。
    assert direct.status_code == 200
    assert direct.json()["id"] == "不是整数"
    assert "internal_note" in direct.json()

# JSONResponse 接收 JSON 兼容对象；Response 可接收已生成的 JSON 文本。
object_response = JSONResponse(content=dated.model_dump(mode="json"))
text_response = Response(content=dated.model_dump_json(), media_type="application/json")
same_content = json.loads(object_response.body) == json.loads(text_response.body)
print("两种显式响应的内容相同：", same_content)  # 预期：两种显式响应的内容相同： True。
assert same_content

200 {'id': '不是整数', 'internal_note': '虚构演示字段'}
两种显式响应的内容相同： True


## 本章小结

（1）输出模型说明公开结构；输入字段和内部字段不必全部出现在响应中。

（2）404 表达查询不到记录，422 表达请求校验失败；不合格的响应是需要修正代码的 500 程序错误。

（3）处理器负责实际错误内容，responses 负责文档声明；两者与示例数据必须一起核对。

（4）设置 Response 参数的响应头仍保留模型处理；直接返回 Response 对象则需要自行保证其内容符合契约。

理解自查：相同的 response_model 声明下，普通字典和 JSONResponse 为什么可能产生不同的公开字段？

## 练习

（1）给内部记录增加 owner_note，保持输出模型不变。核对成功查询与创建响应，确认两个响应都没有 owner_note，也没有 private_note。

In [14]:
# 在原 records 和创建逻辑里加入 owner_note，保持 RecordOutput 不变。
# 创建、查询后同时观察内部字典和公开 JSON。

（2）将 contract_app 的请求校验错误码改为 BAD_INPUT，并同步更新 422 文档示例。发送非整数编号，确认状态码仍为 422，实际 JSON 和 /openapi.json 中的示例完全一致。

In [15]:
# 重建 contract_app，把处理器和 422 example 同步改为 BAD_INPUT。
# 请求非整数编号，逐项比较状态码、JSON 和 OpenAPI 示例。

（3）修正 /broken，使它满足 RecordOutput；再修正 /direct，改为返回普通字典并保留一个内部字段。确认两个接口均返回 200，公开内容只含 id 与 title。

In [16]:
# 创建本题独立应用，修正 /broken 的字段；让 /direct 返回普通字典。
# 两次请求检查状态码与公开字段；本题不再期待第 8 节的异常。

## 练习提示与解析

以下对应练习（3），先改返回内容与返回方式，再查看。

提示 1：检查 RecordOutput 的两个必填字段。

提示 2：普通数据会经过响应模型，已构造的 Response 对象走另一条处理路径。

### 练习（3）参考解析

让 /broken 返回 {"id": 1, "title": "修正记录"}；让 /direct 返回 {"id": 2, "title": "普通字典", "internal_note": "内部"}，并保留 RecordOutput 声明。返回类型标注也同步为普通字典，避免仍标成 JSONResponse。两次请求均为 200，公开字段均只有 id、title；/direct 的 internal_note 被过滤。修正后不再期待 ResponseValidationError，应把原来的失败检查改成成功内容检查。

## 参考与引用来源

- FastAPI 官方文档：[Response Model - Return Type](https://fastapi.tiangolo.com/tutorial/response-model/) 的返回类型、response_model Priority 与数据过滤小节；[Extra Models](https://fastapi.tiangolo.com/tutorial/extra-models/) 的输入、输出模型分离；[Response Status Code](https://fastapi.tiangolo.com/tutorial/response-status-code/) 与 [Response Headers](https://fastapi.tiangolo.com/advanced/response-headers/) 的状态码与 Response 参数；[Handling Errors](https://fastapi.tiangolo.com/tutorial/handling-errors/) 的 HTTPException、请求校验处理器及 FastAPI/Starlette 异常关系；[Additional Responses in OpenAPI](https://fastapi.tiangolo.com/advanced/additional-responses/) 的 Additional Response with model、Combining information；[Return a Response Directly](https://fastapi.tiangolo.com/advanced/response-directly/) 的直接响应与 JSON 兼容数据；[Testing](https://fastapi.tiangolo.com/tutorial/testing/) 的应用内调用。
- Pydantic 官方文档：[Serialization](https://pydantic.dev/docs/validation/latest/concepts/serialization/) 的 Python mode、JSON mode，支持 model_dump、model_dump(mode="json") 与 model_dump_json 的表示区别。
- Starlette 官方文档：[TestClient](https://starlette.dev/testclient/#testclient) 的应用异常传播与 raise_server_exceptions 参数，支持区分测试代码收到的异常和客户端的 500 响应。
- RFC Editor：[RFC 9110，第 15.3.2 节 201 Created](https://www.rfc-editor.org/rfc/rfc9110.html#section-15.3.2)，支持创建成功响应及 Location 标识新资源的含义。